In [12]:
import pyarrow
print(pyarrow.__version__)

25.0.1


In [13]:
"""
Tennis dataset cleaning pipeline — Q3, Q4, Q5, Q6, Q9, Q11, Q14
Cleans: event, home_team, away_team, home_team_score, away_team_score,
        time, tournament, round

Requires: pandas, pyarrow (pip install pandas pyarrow)
"""

import pandas as pd
import numpy as np

IN_DIR = "./04_clean_tables"          # folder containing the original .parquet files
OUT_DIR = "./cleaned_Yasi"  # folder to write cleaned files into

import os
os.makedirs(OUT_DIR, exist_ok=True)


In [14]:
# ---------------------------------------------------------------------------
# 1. event.parquet — the match spine
# ---------------------------------------------------------------------------
event = pd.read_parquet(f"{IN_DIR}/event.parquet", engine="fastparquet")

# drop exact duplicate matches, if any
event = event.drop_duplicates(subset="match_id").reset_index(drop=True)

# flag matches with no recorded result instead of dropping them
event["match_completed"] = event["winner_code"].notna()

# start_datetime is a Unix timestamp (seconds) -> readable datetime + month bucket
event["start_datetime_utc"] = pd.to_datetime(event["start_datetime"], unit="s", utc=True)
event["start_month"] = event["start_datetime_utc"].dt.strftime("%Y-%m")

event.to_parquet(f"{OUT_DIR}/event.parquet", index=False)

In [15]:
# ---------------------------------------------------------------------------
# 2 & 3. home_team.parquet / away_team.parquet — player info per match
# ---------------------------------------------------------------------------
def clean_team(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates(subset="match_id").reset_index(drop=True)

    # trim stray whitespace on text fields
    text_cols = [c for c in ["name", "slug", "residence", "birthplace", "plays",
                         "country", "name_code", "full_name", "gender"] if c in df.columns]

    for c in text_cols:
            df[c] = df[c].astype("string").str.strip()

    # turned_pro was stored as a string year -> nullable integer
    df["turned_pro"] = pd.to_numeric(df["turned_pro"], errors="coerce").astype("Int64")

    # flag (don't remove) physically implausible values
    df["weight_flag"] = ((df["weight"] < 40) | (df["weight"] > 150)).fillna(False)

    return df

home_team = clean_team(pd.read_parquet(f"{IN_DIR}/home_team.parquet", engine="fastparquet"))
away_team = clean_team(pd.read_parquet(f"{IN_DIR}/away_team.parquet", engine="fastparquet"))

home_team.to_parquet(f"{OUT_DIR}/home_team.parquet", index=False)
away_team.to_parquet(f"{OUT_DIR}/away_team.parquet", index=False)

# NOTE: home_team/away_team have fewer rows than event (12,389 / 11,690 vs 16,873).
# Those matches simply have no player row recorded — a structural gap, not
# something a null-fill can fix. Use a LEFT join from event when combining.

In [16]:
# ---------------------------------------------------------------------------
# 4 & 5. home_team_score.parquet / away_team_score.parquet — set scores
# ---------------------------------------------------------------------------
def clean_score(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates(subset="match_id").reset_index(drop=True)

    # drop columns that are 100% empty (no 5-setters in this dataset)
    dead_cols = ["period_4", "period_5",
                 "period_4_tie_break", "period_5_tie_break", "normal_time"]
    dead_cols = [c for c in dead_cols if c in df.columns and df[c].isna().all()]
    df = df.drop(columns=dead_cols)

    # count sets actually played per match
    set_cols = [c for c in ["period_1", "period_2", "period_3"] if c in df.columns]
    df["sets_played"] = df[set_cols].notna().sum(axis=1).astype("Int64")
    df.loc[df[set_cols].isna().all(axis=1), "sets_played"] = pd.NA

    return df

home_team_score = clean_score(pd.read_parquet(f"{IN_DIR}/home_team_score.parquet", engine="fastparquet"))
away_team_score = clean_score(pd.read_parquet(f"{IN_DIR}/away_team_score.parquet", engine="fastparquet"))

home_team_score.to_parquet(f"{OUT_DIR}/home_team_score.parquet", index=False)
away_team_score.to_parquet(f"{OUT_DIR}/away_team_score.parquet", index=False)

In [17]:
# ---------------------------------------------------------------------------
# 6. time.parquet — set/match timing
# ---------------------------------------------------------------------------
time_df = pd.read_parquet(f"{IN_DIR}/time.parquet", engine="fastparquet")
time_df = time_df.drop_duplicates(subset="match_id").reset_index(drop=True)

# drop 100%-empty columns
dead_cols = [c for c in ["period_4", "period_5"] if c in time_df.columns and time_df[c].isna().all()]
time_df = time_df.drop(columns=dead_cols)

period_cols = [c for c in ["period_1", "period_2", "period_3"] if c in time_df.columns]

# --- bounds, chosen from the actual distribution + tennis domain knowledge ---
MIN_PLAUSIBLE_SET_SECONDS = 300     # 5 min: no completed set finishes faster
MAX_PLAUSIBLE_SET_SECONDS = 14400   # 4 h: 99.9th pctile is far below this; beyond it is corrupted data, not a long set

clean_periods = time_df[period_cols].copy()
issue_mask = pd.Series(False, index=time_df.index)
for c in period_cols:
    bad = (clean_periods[c] < MIN_PLAUSIBLE_SET_SECONDS) | (clean_periods[c] > MAX_PLAUSIBLE_SET_SECONDS)
    issue_mask |= bad.fillna(False)
    clean_periods.loc[bad, c] = np.nan

# --- cross-check against the score tables: does a recorded set duration exist
#     wherever the score table says a set was actually played? ---
sets_played = home_team_score[['match_id']].copy()
set_cols_score = [c for c in ['period_1','period_2','period_3'] if c in home_team_score.columns]
sets_played['sets_played'] = home_team_score[set_cols_score].notna().sum(axis=1)

time_df = time_df.merge(sets_played, on='match_id', how='left')

incomplete_mask = pd.Series(False, index=time_df.index)
for i, c in enumerate(period_cols, start=1):
    should_exist = time_df['sets_played'] >= i
    missing = clean_periods[c].isna() & should_exist
    incomplete_mask |= missing.fillna(False)

for c in period_cols:
    time_df[c] = clean_periods[c]

time_df['time_data_issue'] = issue_mask
time_df['time_incomplete'] = incomplete_mask
time_df['duration_seconds'] = clean_periods[period_cols].sum(axis=1, min_count=1)
time_df['duration_minutes'] = (time_df['duration_seconds'] / 60).round(1)
time_df['duration_reliable'] = ~(issue_mask | incomplete_mask)


time_df = time_df.drop(columns=['sets_played'])
time_df.to_parquet(f"{OUT_DIR}/time.parquet", index=False)

In [18]:
# ---------------------------------------------------------------------------
# 7. tournament.parquet — surface, category, tournament identity
# ---------------------------------------------------------------------------
tournament = pd.read_parquet(f"{IN_DIR}/tournament.parquet", engine="fastparquet")
tournament = tournament.drop_duplicates(subset="match_id").reset_index(drop=True)

# tournament_unique_id is 100% null in this dataset -> useless, drop it
cols_to_drop = [c for c in ["tournament_unique_id"] if c in tournament.columns]
tournament = tournament.drop(columns=cols_to_drop)

text_cols = ["tournament_name", "tournament_slug",
             "tournament_category_name", "tournament_category_slug", "ground_type"]
for c in text_cols:
    if c in tournament.columns:
        tournament[c] = tournament[c].astype("string").str.strip()

tournament.to_parquet(f"{OUT_DIR}/tournament.parquet", index=False)

# NOTE: since tournament_unique_id is unusable, group by
# (tournament_name, tournament_category_name) + event.start_month
# to identify distinct tournament editions for Q9.

In [19]:
# ---------------------------------------------------------------------------
# 8. round.parquet — round of each match
# ---------------------------------------------------------------------------
round_df = pd.read_parquet(f"{IN_DIR}/round.parquet", engine="fastparquet")
round_df = round_df.drop_duplicates(subset="match_id").reset_index(drop=True)

# fix inconsistent labels found in the data: singular vs plural variants
name_map = {"Quarterfinal": "Quarterfinals", "Semifinal": "Semifinals"}
slug_map = {"quarterfinal": "quarterfinals", "semifinal": "semifinals"}
round_df["name"] = round_df["name"].replace(name_map)
round_df["slug"] = round_df["slug"].replace(slug_map)

round_df["is_final"] = round_df["name"].eq("Final")

round_df.to_parquet(f"{OUT_DIR}/round.parquet", index=False)

# NOTE: round is only present for 9,243 of 16,873 matches. That ~45% gap
# is structural (missing at the source) and will limit which matches
# can be classified as tournament finals for Q9.


In [20]:
import pandas as pd
t = pd.read_parquet("cleaned_Yasi/tournament.parquet")
print(t["ground_type"].value_counts())

ground_type
Hardcourt outdoor    8116
Red clay             5609
Hardcourt indoor     2172
Grass                 242
Red clay indoor       218
Carpet indoor         126
Synthetic outdoor      99
Green clay             18
Name: count, dtype: int64[pyarrow]


In [21]:
t["ground_type"] = t["ground_type"].str.strip().str.title()

In [22]:
print("Done. Cleaned files written to:", OUT_DIR)

Done. Cleaned files written to: ./cleaned_Yasi
